Датасет содержит 251 строку о книгах: название, автор, жанр, средняя оценка, возрастная рекомендация и метрики популярности — сколько раз книгу добавили в избранное, прочитали, процитировали, отрецензировали и запланировали к прочтению.

**Главный вопрос эссе:** какие факторы связаны с популярностью книги ?  
**Подвопросы:**
1. Как распределены оценки книг ?
2. Отличаются ли метрики популярности в разных возрастных категориях ?
3. Связаны ли между собой разные метрики популярности (прочтения, избранное, рецензии) ?
4. Какие авторы и жанры самые популярные?

**Гипотеза:** высокая оценка книги положительно связана с числом прочтений и добавлений в избранное, а жанр (фэнтези, приключения) — с более широкой аудиторией.

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from ipywidgets import interact, widgets
import ipywidgets as widgets

In [35]:
df_raw = pd.read_csv('books.csv', encoding='utf-8-sig')

## Первое знакомство с данными

In [27]:
print("Размер таблицы:", df_raw.shape)
print("\nКолонки:", df_raw.columns.tolist())

display(df_raw.head())

df_raw.info()

display(df_raw.describe())

Размер таблицы: (251, 11)

Колонки: ['Название', 'code', 'Оценка', 'Количество в избранном', 'Количество цитат', 'Количество прочитано', 'Количество рецензий', 'Количество планируют', 'Автор', 'Жанр', 'Возраст']


,Название,code,Оценка,Количество в избранном,Количество цитат,Количество прочитано,Количество рецензий,Количество планируют,Автор,Жанр,Возраст
0,101 далматинец,1,4.30,9,1.0,354.0,16.0,221.0,Доди Смит,зарубежное,7
1,"Айван, единственный и неповторимый",3,4.40,5,23.0,92.0,13.0,167.0,Кэтрин Эпплгейт,зарубежное,8
2,"Айван, единственный и неповторимый",3,4.40,5,23.0,92.0,13.0,167.0,Кэтрин Эпплгейт,приключения,8
3,"Алиса в Стране чудес и ""Алиса в Зазеркалье""",4,4.32,349,1079.0,8830.0,535.0,2228.0,Льюис Кэрролл,зарубежное,9
4,"Алиса в Стране чудес и ""Алиса в Зазеркалье""",4,4.32,349,1079.0,8830.0,535.0,2228.0,Льюис Кэрролл,классика,9


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251 entries, 0 to 250
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Название                251 non-null    object 
 1   code                    251 non-null    int64  
 2   Оценка                  251 non-null    float64
 3   Количество в избранном  251 non-null    int64  
 4   Количество цитат        251 non-null    float64
 5   Количество прочитано    251 non-null    float64
 6   Количество рецензий     251 non-null    float64
 7   Количество планируют    251 non-null    float64
 8   Автор                   251 non-null    object 
 9   Жанр                    251 non-null    object 
 10  Возраст                 251 non-null    int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 21.7+ KB


,code,Оценка,Количество в избранном,Количество цитат,Количество прочитано,Количество рецензий,Количество планируют,Возраст
count,251.000000,251.000000,251.000000,251.000000,251.000000,251.000000,251.000000,251.000000
mean,46.079681,4.315478,163.764940,188.705179,4133.187251,191.119522,1254.282869,7.418327
std,24.617913,0.438958,491.453612,312.585209,10357.872122,242.189615,1895.181054,1.421374
min,1.000000,2.620000,0.000000,0.000000,3.000000,0.000000,9.000000,5.000000
25%,24.500000,4.245000,7.000000,14.000000,327.500000,18.000000,133.000000,7.000000
50%,46.000000,4.410000,29.000000,49.000000,811.000000,79.000000,364.000000,7.000000
75%,65.000000,4.580000,133.000000,267.000000,4718.000000,334.500000,1626.000000,8.000000
max,90.000000,5.000000,3572.000000,1951.000000,75985.000000,1274.000000,10617.000000,10.000000


В таблице 251 строка и 11 колонок.

Сразу видно, что книги повторяются например, «Алиса в Стране чудес и "Алиса в Зазеркалье"» встречается минимум 2 раза с одинаковыми метриками, но разными жанрами, что означает, что одна строка это одна пара (книга, жанр), а не одна книга.

Колонки делятся на:
- **Идентификаторы:** Название, code
- **Метрики:** Оценка, Количество в избранном/цитат/прочитано/рецензий/планируют
- **Категории:** Автор, Жанр, Возраст

Пропуски: все числовые колонки заполнены (в *info()* видно, что *Non-Null Count* = 251 для всех). Но есть дубликаты по *code* — с ними и будем работать

In [28]:
print("Всего строк:", len(df_raw))
print("Уникальных книг (по code):", df_raw['code'].nunique())
print("Уникальных названий:", df_raw['Название'].nunique())
print("Уникальных авторов:", df_raw['Автор'].nunique())
print("Уникальных жанров:", df_raw['Жанр'].nunique())

print("\nПропуски по колонкам:")
print(df_raw.isna().sum())

print("\nСколько раз встречается каждая книга (топ-5 по числу жанров):")
print(df_raw.groupby('code').size().sort_values(ascending=False).head())

print("\nТоп-10 жанров по числу книг:")
print(df_raw.groupby('Жанр')['code'].nunique().sort_values(ascending=False).head(10))

Всего строк: 251
Уникальных книг (по code): 87
Уникальных названий: 87
Уникальных авторов: 74
Уникальных жанров: 19

Пропуски по колонкам:
Название                  0
code                      0
Оценка                    0
Количество в избранном    0
Количество цитат          0
Количество прочитано      0
Количество рецензий       0
Количество планируют      0
Автор                     0
Жанр                      0
Возраст                   0
dtype: int64

Сколько раз встречается каждая книга (топ-5 по числу жанров):
code
21    6
54    6
39    6
60    6
57    5
dtype: int64

Топ-10 жанров по числу книг:
Жанр
зарубежное      55
проза           30
сказки          29
русское         24
приключения     24
классика        20
20 век          10
подростковое    10
фэнтези          9
фантастика       9
Name: code, dtype: int64


В данных 251 строк, но всего 87 уникальных книг. 

Разница за счёт множественных жанров у одной книги. К примеру, у "Гарри Поттера" 4 жанра (приключения, фэнтези, зарубежное, подростковое), у "Алисы в Стране чудес" - 3. 

Пропусков в данных нет. 

Самые частые жанры - зарубежное, проза, сказки, русское и приключения.

## Очистка и предобработка

Поскольку одна книга встречается несколько раз (по числу жанров), для анализа на уровне книги создадим отдельный датафрейм *books*, где одна строка = одна книга. Для анализа жанров будем использовать исходный *df_raw*. Пропусков нет, но приведём числовые колонки к *int* (они пришли как *float* из-за десятичных значений в "Оценке"), а "Возраст" сделаем категориальной переменной.

In [29]:
# 1. Одна строка = одна книга (для анализа метрик и авторов)
books = df_raw.drop_duplicates(subset='code').copy()

# 2. Приводим счётчики к int
count_cols = ['Количество в избранном', 'Количество цитат', 'Количество прочитано',
              'Количество рецензий', 'Количество планируют']
for col in count_cols:
    books[col] = books[col].fillna(0).astype(int)

# 3. Возраст — категория
books['Возраст'] = books['Возраст'].astype(int).astype('category')

print("После удаление дублей:", books.shape)
display(books.head(3))

После удаление дублей: (87, 11)


,Название,code,Оценка,Количество в избранном,Количество цитат,Количество прочитано,Количество рецензий,Количество планируют,Автор,Жанр,Возраст
0,101 далматинец,1,4.30,9,1,354,16,221,Доди Смит,зарубежное,7
1,"Айван, единственный и неповторимый",3,4.40,5,23,92,13,167,Кэтрин Эпплгейт,зарубежное,8
3,"Алиса в Стране чудес и ""Алиса в Зазеркалье""",4,4.32,349,1079,8830,535,2228,Льюис Кэрролл,зарубежное,9


## 1. Как распределены оценки книг?

In [30]:
def plot_rating_hist(nbins=15):
    fig = px.histogram(books, x='Оценка', nbins=nbins,
                       title=f'Распределение оценок книг (бинов: {nbins})',
                       labels={'Оценка': 'Средняя оценка', 'count': 'Число книг'})
    fig.update_layout(bargap=0.05)
    fig.show()

interact(plot_rating_hist,
         nbins=widgets.IntSlider(min=5, max=40, step=1, value=15, description='Бинов:'))

interactive(children=(IntSlider(value=15, description='Бинов:', max=40, min=5), Output()), _dom_classes=('widg…

<function __main__.plot_rating_hist(nbins=15)>

Распределение оценок - левостороннее с пиком около 4.5.

Большинство книг имеют оценки 4.2-4.7, очень мало книг ниже 4.0. 

Это типично для платформ с отзывами: аудитория ставит высокие оценки любимым книгам, а низкие - редко. Значит, оценка сама по себе плохо разделяет книги - нужны метрики популярности.

## 2. Популярность по возрастным категориям

In [31]:
def plot_by_age(metric='Количество прочитано'):
    fig = px.box(books, x='Возраст', y=metric, color='Возраст',
                 title=f'«{metric}» по возрастным категориям',
                 points='all')
    fig.update_layout(showlegend=False)
    fig.show()

interact(plot_by_age,
         metric=widgets.Dropdown(
             options=['Количество прочитано', 'Количество в избранном',
                      'Количество рецензий', 'Количество планируют'],
             value='Количество прочитано',
             description='Метрика:'))

interactive(children=(Dropdown(description='Метрика:', options=('Количество прочитано', 'Количество в избранно…

<function __main__.plot_by_age(metric='Количество прочитано')>

Медиана по всем возрастным категориям невелика ( примерно 300–1500 прочтений), но разброс огромен: у книг для возраста 8–9 есть выбросы в десятки тысяч (Гарри Поттер, Хоббит, Маленький принц). Возраст 5–6 в среднем читают меньше, но и книг в этой категории немного. 

Вывод: не возраст определяет популярность, а конкретная книга. 

Если переключить метрику на «Количество в избранном» — картина та же: несколько книг далеко впереди.

## 3. Связаны ли метрики популярности между собой?

In [32]:
def plot_scatter(x_metric, y_metric, color_metric):
    fig = px.scatter(books, x=x_metric, y=y_metric,
                     color=color_metric,
                     hover_name='Название',
                     log_x=True, log_y=True,
                     title=f'{y_metric} vs {x_metric}')
    fig.show()

interact(plot_scatter,
         x_metric=widgets.Dropdown(
             options=['Количество прочитано', 'Количество в избранном', 'Количество рецензий'],
             value='Количество прочитано', description='X:'),
         y_metric=widgets.Dropdown(
             options=['Количество прочитано', 'Количество в избранном', 'Количество рецензий'],
             value='Количество рецензий', description='Y:'),
         color_metric=widgets.Dropdown(
             options=['Возраст', 'Оценка'],
             value='Возраст', description='Цвет:'))

interactive(children=(Dropdown(description='X:', options=('Количество прочитано', 'Количество в избранном', 'К…

<function __main__.plot_scatter(x_metric, y_metric, color_metric)>

Точки вытянуты вдоль прямой - метрики сильно коррелируют: книги, которые много читают, чаще добавляют в избранное и чаще рецензируют.

Наведите курсор на верхние точки - это Гарри Поттер, Хоббит, Маленький принц, Таинственный сад. 

Если переключить цвет с "Возраст" на "Оценка" - картина не изменится: и возраст, и оценка своего рода размазаны по всему облаку, что значит, что популярность не сводится ни к возрастной категории, ни к высокой оценке.

Переключите Y на "Количество в избранном" - картинка не измменится.

## 4.1. Кто самые популярные авторы?

In [33]:
def plot_top_authors(top_n=10, metric='Количество прочитано'):
    top = (books.groupby('Автор')[metric]
                .sum()
                .nlargest(top_n)
                .reset_index())
    fig = px.bar(top, x=metric, y='Автор', orientation='h',
                 title=f'Топ-{top_n} авторов по «{metric}» (сумма)')
    fig.update_layout(yaxis={'categoryorder': 'total ascending'})
    fig.show()

interact(plot_top_authors,
         top_n=widgets.IntSlider(min=5, max=30, step=1, value=10, description='Топ-N:'),
         metric=widgets.Dropdown(
             options=['Количество прочитано', 'Количество в избранном',
                      'Количество рецензий', 'Количество планируют'],
             value='Количество прочитано', description='Метрика:'))

interactive(children=(IntSlider(value=10, description='Топ-N:', max=30, min=5), Dropdown(description='Метрика:…

<function __main__.plot_top_authors(top_n=10, metric='Количество прочитано')>

По прочтениям лидируют Джоан Роулинг, Дж. Р. Р. Толкин, Антуан де Сент-Экзюпери - их книги дают десятки тысяч прочтений.

По избранному картина похожа. Но если переключить на "Количество рецензий" - на первое место выйдет Антуан де Сент-Экзюпери, а Жюль Верн поднимется на второе. Рецензий много у книг с высокой эмоциональной вовлечённостью, а не только у блокбастеров. При уменьшении топа до 5 - видно, что "длинный хвост" авторов почти не даёт вклада.

## 4.2. Какие жанры популярнее?

In [34]:
def plot_genres(top_n=10, min_books=3):
    stats = (df_raw.groupby('Жанр')
                    .agg(books_count=('code', 'nunique'),
                         avg_rating=('Оценка', 'mean'),
                         total_reads=('Количество прочитано', 'sum'))
                    .reset_index())
    stats = stats[stats['books_count'] >= min_books]
    top = stats.nlargest(top_n, 'total_reads')
    fig = px.bar(top, x='total_reads', y='Жанр', orientation='h',
                 hover_data=['books_count', 'avg_rating'],
                 title=f'Топ-{top_n} жанров по числу прочтений (книг в жанре ≥ {min_books})')
    fig.update_layout(yaxis={'categoryorder': 'total ascending'})
    fig.show()

interact(plot_genres,
         top_n=widgets.IntSlider(min=3, max=20, step=1, value=10, description='Топ-N:'),
         min_books=widgets.IntSlider(min=1, max=10, step=1, value=3, description='Мин. книг:'))

interactive(children=(IntSlider(value=10, description='Топ-N:', max=20, min=3), IntSlider(value=3, description…

<function __main__.plot_genres(top_n=10, min_books=3)>

Суммарно лидируют "зарубежное", "фэнтези", "приключения" - но это отчасти следствие того, что это самые частые теги. Если посмотреть *avg_rating* при наведении, у "фэнтези" и "приключений" средние оценки не ниже, но книг в них меньше — зато среди них блокбастеры (Гарри Поттер, Хоббит). То есть жанр не гарантирует популярность, но "фэнтези" концентрирует книги-хиты.

## Заключение

В ходе анализа мы выяснили:

1. **Данные многоуровневые.** Одна книга встречается столько раз, сколько у неё жанров. Для анализа на уровне книги нужно исключать дубли по полю *code*;
2. **Оценки смещены вправо.** Большинство книг имеют оценку 4.2–4.7, поэтому сама по себе оценка плохо дифференцирует книги;
3. **Метрики популярности сильно коррелируют.** Книги, которые много читают, чаще добавляют в избранное и рецензируют;
4. **Возрастная категория не определяет популярность.** Разброс внутри каждой категории огромен, а выбросы (Гарри Поттер, Хоббит, Маленький принц) сидят в нескольких возрастах;
5. **Популярность концентрируется.** Топ-5 авторов дают непропорционально большую долю прочтений;
6. **Жанр сам по себе - не драйвер.** Но "фэнтези" и "приключения" концентрируют книги-хиты.

**Гипотеза частично подтвердилась:** высокая оценка связана с популярностью, но не объясняет её целиком. Решающую роль играют конкретные книги/авторы, а не категории.